# Nightingale: the transformer's full report (EXP-021), on Kaggle

D-13 has already decided, on seed 42, that the small transformer runs the pipeline (2026-10-13). This
notebook produces the rest of what EXP-021 promised, on the CPU, in the background:
- every seed (42–46) of both transformer arms;
- against the logistic regressions and XGBoost models trained like them;
- at 100%, 50% and 25% of the history;
- with McNemar's test and the paired bootstraps, and D-13's first two checks at every seed.

The golden cases were already checked on the laptop, for all ten networks. **Nothing is trained,
and the test split is never downloaded.**

## How to run it (about 5 minutes of your time)

1. **Once:** on kaggle.com, **Create → New Dataset**:
   - upload `models/kaggle_upload/nightingale_models.zip` from the laptop (about 50 MB);
   - name it `nightingale-models`;
   - keep it **Private**: models trained on DDXPlus are never published.
2. **Create → New Notebook → File → Import Notebook**: upload this file.
3. **Add Input → Your Datasets → nightingale-models.**
4. **Session options:**
   - **Accelerator: None.** The CPU is enough: the laptop's NumPy forward pass is what gets
     scored.
   - **Internet: on.** It clones the repository and downloads DDXPlus's validation split.
5. **Save Version → Save & Run All (Commit) → Save**, then close the browser.
   - It should take about 30–60 minutes. This is an estimate: nothing has run on Kaggle's CPU yet.
6. When Kaggle emails you, open the latest version → **Output**:
   - download **`exp021_reports.zip`**;
   - unzip it into `data/interim/` on the laptop, and tell Claude.


In [ ]:
COMMIT = "master"
SEEDS = (42, 43, 44, 45, 46)

In [ ]:
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

REPO = Path("/tmp/nightingale")  # outside /kaggle/working, so no data ends up in the output
OUT = Path("/kaggle/working/exp021_reports")
OUT.mkdir(parents=True, exist_ok=True)


def run(*command, **kwargs):
    print("$", " ".join(map(str, command)), flush=True)
    subprocess.run([str(c) for c in command], check=True, **kwargs)


if not REPO.exists():
    run("git", "clone", "-q", "https://github.com/HarshRohila02/nightingale.git", REPO)
run("git", "checkout", "-q", COMMIT, cwd=REPO)
run("git", "log", "--oneline", "-1", cwd=REPO)
# The comparators were trained with XGBoost 2.1.4 (their run.json); score them with the same.
run(sys.executable, "-m", "pip", "install", "-q", "xgboost==2.1.4")

In [ ]:
# The models, from the private dataset added as input (found by folder name, zipped or not).
sys.path.insert(0, str(REPO / "scripts"))
sys.path.insert(0, str(REPO))
from report_deep import BUNDLES, place_models

found = place_models(Path("/kaggle/input"), REPO / "models")
missing = [b for b in BUNDLES if not (REPO / "models" / b).exists()]
print("placed:", found)
assert not missing, f"not found in /kaggle/input: {missing}. Add Input -> Your Datasets -> nightingale-models"

In [ ]:
# DDXPlus's validation split, at the snapshot the laptop uses (docs/11 §4).
# test.csv is never fetched before Phase 4 (docs/05 §2), and train.csv is not needed here.
from huggingface_hub import hf_hub_download

for name in ["release_evidences.json", "release_conditions.json", "validate.csv"]:
    hf_hub_download(
        "aai530-group6/ddxplus",
        name,
        repo_type="dataset",
        revision="2ad986acc1ec62fb4a94171acc43f4fdd5bfde53",
        local_dir=str(REPO / "data" / "raw" / "ddxplus"),
    )
run(sys.executable, "scripts/decode_ddxplus.py", cwd=REPO)
run(sys.executable, "scripts/build_ddxplus_chestpain.py", "--split", "validate", cwd=REPO)

In [ ]:
# Smoke test: 300 patients, one seed. A failure stops the version here. Its figures mean nothing.
run(sys.executable, "scripts/report_deep.py", "--seeds", "42", "--limit", "300", "--no-golden",
    "--out", "/tmp/smoke.json", cwd=REPO)

In [ ]:
# The report: every seed, both arms, every comparator, every level. Its log goes to the output.
started = time.time()
with open(OUT / "report.log", "w", encoding="utf-8") as handle:
    result = subprocess.run(
        [sys.executable, "scripts/report_deep.py", "--seeds", *map(str, SEEDS), "--no-golden",
         "--out", str(OUT / "exp021_report.json")],
        cwd=REPO, stdout=handle, stderr=subprocess.STDOUT,
    )
minutes = (time.time() - started) / 60
status = "done" if result.returncode == 0 else f"FAILED (exit {result.returncode})"
print(f"report {status} after {minutes:.0f} min")
print(open(OUT / "report.log", encoding="utf-8").read()[-3000:])

In [ ]:
# What finished, and the one file to download (Output tab: exp021_reports.zip).
(OUT / "STATUS.txt").write_text(f"report: {status}, {minutes:.0f} min\n", encoding="utf-8")
archive = shutil.make_archive("/kaggle/working/exp021_reports", "zip", OUT)
print(archive, f"{os.path.getsize(archive) / 1e6:.1f} MB")